In [21]:
import pandas as pd
import numpy as np
from IPython.display import display
from google.colab import drive
import duckdb

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
#connection
db_path = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/staging.duckdb"
conn = duckdb.connect(db_path)

tables = conn.execute("SHOW TABLES").df()
tables

,name
0,mortality_2018
1,mortality_2019
2,mortality_2020
3,mortality_2021
4,mortality_2022
5,mortality_2023
6,mortality_2024


In [23]:
#extracting data for:
#I42 (Cardiomyopathy), I25 (Chronic Ischemic Heart Disease), I21 (Heart Attack), I50 (Heart Failure)

extract_query = ""
for i in range(2018, 2025):
  extract_query += f"SELECT * FROM mortality_{i} WHERE cause LIKE 'I42%' OR cause LIKE 'I25%' OR cause LIKE 'I21%' OR cause LIKE 'I50%'"
  if i != 2024:
    extract_query += "\nUNION ALL\n"
df = conn.execute(extract_query).df()
conn.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
#relabelling sex
sex_map = {
    "M": "Male",
    "F": "Female"
}

df["sex"] = df['sex'].map(sex_map)
df.head()

,year,month,sex,age,race,cause
0,2018,01,Male,1081,01,I500
1,2018,01,Female,1089,01,I255
2,2018,01,Female,1087,01,I250
3,2018,01,Male,1076,03,I500
4,2018,01,Male,1062,01,I219


In [25]:
#extracting the age
df["age_numeric"] = pd.to_numeric(df["age"].str[1:], errors="coerce")
age_condition = [
    (df["age"].str.startswith("1")) & (df["age_numeric"] <= 24),
    (df['age'].str.startswith("1")) & (df['age_numeric'] >=25) & (df['age_numeric'] <= 44),
    (df['age'].str.startswith("1")) & (df['age_numeric'] >=45) & (df['age_numeric']<=64),
    (df["age"].str.startswith("1")) & (df["age_numeric"] >=65)
]

age_choices = [
    "0-24",
    "25-44",
    "45-64",
    "65+"
]

df["age_group"] = np.select(age_condition, age_choices, default="Exclude")
final_df = df[df["age_group"] != "Exclude"].copy()
final_df.head()

,year,month,sex,age,race,cause,age_numeric,age_group
0,2018,01,Male,1081,01,I500,81,65+
1,2018,01,Female,1089,01,I255,89,65+
2,2018,01,Female,1087,01,I250,87,65+
3,2018,01,Male,1076,03,I500,76,65+
4,2018,01,Male,1062,01,I219,62,45-64


In [26]:
#mapping for cause of death
cause_mapping = {
    "I42": "Cardiomyopathy",
    "I25": "Chronic Ischemic Heart Disease",
    "I21": "Heart Attack",
    "I50": "Heart Failure"
}

final_df["cause_of_death"] = final_df["cause"].str[:3].map(cause_mapping)
final_df.head()

,year,month,sex,age,race,cause,age_numeric,age_group,cause_of_death
0,2018,01,Male,1081,01,I500,81,65+,Heart Failure
1,2018,01,Female,1089,01,I255,89,65+,Chronic Ischemic Heart Disease
2,2018,01,Female,1087,01,I250,87,65+,Chronic Ischemic Heart Disease
3,2018,01,Male,1076,03,I500,76,65+,Heart Failure
4,2018,01,Male,1062,01,I219,62,45-64,Heart Attack


In [27]:
# remove columns
final_df = final_df.drop(columns=["month", "age", "cause", "race", "age_numeric", "sex"])
final_df["year"] = final_df["year"].astype(int)
final_df.head()

,year,age_group,cause_of_death
0,2018,65+,Heart Failure
1,2018,65+,Chronic Ischemic Heart Disease
2,2018,65+,Chronic Ischemic Heart Disease
3,2018,65+,Heart Failure
4,2018,45-64,Heart Attack


In [28]:
#groupping the column for counting death per diseases
death_per_disease = final_df.groupby(["year", "age_group", "cause_of_death"]).size().reset_index(name="deaths")

#loading population table
popln_path = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/clean_age_popln.csv"
popln_df = pd.read_csv(popln_path)


all_disease_comparison = pd.merge(
    death_per_disease,
    popln_df,
    on=["year", "age_group"],
    how="inner"
)

all_disease_comparison["mortality_rate_per_100k"] = ((all_disease_comparison['deaths']/ all_disease_comparison["population"])*100000).round(1)
display(all_disease_comparison.head(4)) #export
print()

#just for comparision
preview = all_disease_comparison.groupby(["cause_of_death", "age_group"]).sum()["deaths"]
display(preview)



,year,age_group,cause_of_death,deaths,population,mortality_rate_per_100k
0,2018,0-24,Cardiomyopathy,254,104008592.0,0.2
1,2018,0-24,Chronic Ischemic Heart Disease,70,104008592.0,0.1
2,2018,0-24,Heart Attack,57,104008592.0,0.1
3,2018,0-24,Heart Failure,61,104008592.0,0.1


cause_of_death                  age_group
Cardiomyopathy                  0-24            1713
                                25-44          10367
                                45-64          33920
                                65+            93236
Chronic Ischemic Heart Disease  0-24             477
                                25-44          24844
                                45-64         301849
                                65+          1477294
Heart Attack                    0-24             337
                                25-44          15114
                                45-64         160046
                                65+           544933
Heart Failure                   0-24             429
                                25-44           5916
                                45-64          51452
                                65+           553257
Name: deaths, dtype: int64

In [29]:
#state level data extraction
state_df = pd.read_csv("/content/drive/MyDrive/mapping_mortality_gap/raw_data/wonder_exports/landscape_state_year.csv")
state_df["Year"] = pd.to_numeric(state_df["Year"], errors="coerce")
state_df = state_df.dropna(subset="Year")
state_df["Year"] = state_df["Year"].astype(int)
state_df = state_df.rename(columns={
    "Residence State (without Puerto Rico)": "state",
    "Year": "year",
    "Deaths": "deaths",
    "Population": "population"

})

columns_todrop = [
    "Notes",
    "Residence State (without Puerto Rico) Code",
    "Year Code",
    "Crude Rate",
    "Crude Rate Lower 95% Confidence Interval",
    "Crude Rate Upper 95% Confidence Interval",
]

state_df = state_df.drop(columns=columns_todrop)

final_state_df = state_df.groupby(["year", "state"]).agg(["sum"])

#calculating rate:
final_state_df["mortality_rate_per_100k"] = ((final_state_df["deaths"] / final_state_df["population"]) * 100000).round(1)
final_state_df = final_state_df.reset_index()

#flattening the colums for visualization (there came popln sum, death sum like those it gets removed)
final_state_df.columns = ["year", "state", "deaths", "population", "mortality_rate_per_100k"]
final_state_df.head() #export



,year,state,deaths,population,mortality_rate_per_100k
0,2018,Alabama,13473.0,4887871.0,275.6
1,2018,Alaska,815.0,737438.0,110.5
2,2018,Arizona,12455.0,7171646.0,173.7
3,2018,Arkansas,8171.0,3013825.0,271.1
4,2018,California,62547.0,39557045.0,158.1


In [30]:
# Cardiomyopathy based on all 50 US state & gender specific (young population only)

deepdive_df = pd.read_csv("/content/drive/MyDrive/mapping_mortality_gap/raw_data/wonder_exports/deepdive_state_sex_year_age.csv")
deepdive_df["Year"] = pd.to_numeric(deepdive_df["Year"], errors="coerce")
deepdive_df = deepdive_df.dropna(subset="Year")
deepdive_df["Year"] = deepdive_df["Year"].astype(int)

columns_todrop = [
    "Residence State (without Puerto Rico) Code",
    "Year Code",
    "Ten-Year Age Groups",
    "Year Code",
    "Sex Code",
    "Crude Rate Lower 95% Confidence Interval",
    "Notes",
    "Crude Rate Upper 95% Confidence Interval",
    "Crude Rate"
]

deepdive_df = deepdive_df.drop(columns=columns_todrop)
deepdive_df = deepdive_df.rename(columns={
    "Residence State (without Puerto Rico)": "state",
    "Year": "year",
    "Sex": "sex",
    "Deaths": "deaths",
    "Population": "population",
    "Ten-Year Age Groups Code": "age_group"
})

# remove the groups for non young population
deepdive_df = deepdive_df.drop(
    deepdive_df[deepdive_df['age_group'].isin(["45-64","65-74", "75-84", "85+"])].index
)

#extracting for state
i42_by_state = deepdive_df.drop(columns=["sex", "age_group"])
i42_by_state = i42_by_state.groupby(["year", "state"]).agg(["sum"])

#finding rate
i42_by_state["mortality_rate_per_100k"] = ((i42_by_state["deaths"] / i42_by_state["population"]) * 100000).round(1)
i42_by_state = i42_by_state.reset_index()

#flattening the columns
i42_by_state.columns = ["year", "state", "deaths", "population", "mortality_rate_per_100k"]
i42_by_state.head() #export




,year,state,deaths,population,mortality_rate_per_100k
0,2018,Alabama,72.0,1246472.0,5.8
1,2018,Arizona,100.0,1717067.0,5.8
2,2018,Arkansas,29.0,366385.0,7.9
3,2018,California,869.0,23848145.0,3.6
4,2018,Colorado,42.0,1056696.0,4.0


In [31]:
#extracting for gender specific
i42_by_sex = deepdive_df.drop(columns=["state", "age_group"])
i42_by_sex = i42_by_sex.groupby(["year", "sex"]).agg(["sum"])

#finding rate
i42_by_sex["mortality_rate_per_100k"] = ((i42_by_sex["deaths"] / i42_by_sex["population"]) * 100000).round(1)
i42_by_sex = i42_by_sex.reset_index()

#flatten the column
i42_by_sex.columns = ["year", "sex", "deaths", "population", "mortality_rate_per_100k"]
i42_by_sex.head() #export

,year,sex,deaths,population,mortality_rate_per_100k
0,2018,Female,1553.0,52087487.0,3.0
1,2018,Male,4321.0,72502949.0,6.0
2,2019,Female,1511.0,47845253.0,3.2
3,2019,Male,4164.0,72305353.0,5.8
4,2020,Female,1440.0,46983203.0,3.1


In [32]:
#Extracting the monthly data from our microdata to see which month is highest for I42
month_data = pd.read_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/national_microdata_clean.csv")
target_age = ["0-24","25-44"]
seasonality = month_data[month_data["age_group"].isin(target_age)].copy()


#drop the columns
seasonality = seasonality.drop(columns=["sex", "cause_of_death", "race_category", "age_group"])

i42_seasonal_spikes = seasonality.groupby(["year", "months"]).size().reset_index(name="deaths")

#make months in proper order

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

i42_seasonal_spikes["months"] = pd.Categorical(
    i42_seasonal_spikes["months"],
    categories=month_order,
    ordered=True
)

i42_seasonal_spikes = i42_seasonal_spikes.sort_values(
    ["year", "months"]
).reset_index(drop=True)

i42_seasonal_spikes.head()

,year,months,deaths
0,2018,January,169
1,2018,February,121
2,2018,March,153
3,2018,April,126
4,2018,May,132


In [33]:
#exporting the data
all_disease_comparison.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/all_HeartDisease_all_age.csv", index=False)
final_state_df.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/all_state_all_HeartDiease_all_age.csv", index=False)
i42_by_state.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/i42_by_state_adults_only.csv", index=False)
i42_by_sex.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/i42_by_sex_adults_only.csv", index=False)
i42_seasonal_spikes.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/i42_seasonal_spikes_by_months.csv", index=False)
